## 01_IMPORTING_LIBRARIES

In [21]:
from pathlib import Path

import re
import unicodedata

import numpy as np
import pandas as pd

from IPython.display import display

## 02_PROJECT_PATHS

In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

DATA_DIR = PROJECT_ROOT / "Data"

TARGET_DIR = DATA_DIR / "Prediction_Targets"

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_DATA_PATH = (
    TARGET_DIR
    / "prediction_targets.csv"
)

print("Prediction target directory:")
print(TARGET_DIR)

Prediction target directory:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets


## 03_LOCKING_THE_PREDICTION_TARGET_LIST

In [5]:
prediction_targets = pd.DataFrame({

    "title": [

        # Released / backtesting cohort
        "Spider-Man: Brand New Day",
        "Michael",
        "The Odyssey",
        "Scary Movie",
        "Mortal Kombat II",
        "The Super Mario Galaxy Movie",
        "The Devil Wears Prada 2",
        "Toy Story 5",
        "GOAT",
        "Backrooms",
        "The Cat in the Hat",

        # Future prediction cohort
        "Dune: Part Three",
        "Avengers: Doomsday",
        "Shrek 5",
        "Jumanji: Open World",
        "Street Fighter",
        "The Batman Part II",
        "Spider-Man: Beyond the Spider-Verse",
        "Children of Blood and Bone"
    ],

    "cohort": [

        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",
        "released_backtest",

        "future_prediction",
        "future_prediction",
        "future_prediction",
        "future_prediction",
        "future_prediction",
        "future_prediction",
        "future_prediction",
        "future_prediction"
    ]
})


print("=" * 80)
print("PREDICTION TARGET REGISTER")
print("=" * 80)

print(
    f"Total target movies: "
    f"{len(prediction_targets)}"
)

print()

print(
    prediction_targets["cohort"]
    .value_counts()
)

display(prediction_targets)

PREDICTION TARGET REGISTER
Total target movies: 19

cohort
released_backtest    11
future_prediction     8
Name: count, dtype: int64


,title,cohort
0,Spider-Man: Brand New Day,released_backtest
1,Michael,released_backtest
2,The Odyssey,released_backtest
3,Scary Movie,released_backtest
4,Mortal Kombat II,released_backtest
5,The Super Mario Galaxy Movie,released_backtest
6,The Devil Wears Prada 2,released_backtest
7,Toy Story 5,released_backtest
8,GOAT,released_backtest
9,Backrooms,released_backtest


## 04 DEFINING THE DATA WE ACTUALLY NEED

In [6]:
target_features = [

    # Identity
    "title",
    "release_date",
    "release_year",

    # Core movie characteristics
    "budget",
    "genre",
    "studio",
    "production_company",

    # Talent
    "director",
    "lead_cast",

    # Franchise information
    "is_franchise",
    "is_sequel",
    "franchise_name",
    "franchise_gap_years",

    # Release strategy
    "release_month",
    "release_season",
    "holiday_release",

    # Pre-release signals
    "pre_release_sentiment",
    "social_interest",

    # Historical strength
    "director_track_record",
    "cast_track_record",

    # Collection metadata
    "data_as_of",
    "source_notes"
]


print("=" * 80)
print("TARGET DATA COLLECTION SCHEMA")
print("=" * 80)

for number, feature in enumerate(
    target_features,
    start=1
):
    print(
        f"{number:02}. {feature}"
    )

TARGET DATA COLLECTION SCHEMA
01. title
02. release_date
03. release_year
04. budget
05. genre
06. studio
07. production_company
08. director
09. lead_cast
10. is_franchise
11. is_sequel
12. franchise_name
13. franchise_gap_years
14. release_month
15. release_season
16. holiday_release
17. pre_release_sentiment
18. social_interest
19. director_track_record
20. cast_track_record
21. data_as_of
22. source_notes


## 05 VERFIED TARGET REGISTER

In [7]:
verified_targets = pd.DataFrame([

    # RELEASED / BACKTEST

    {
        "title": "Spider-Man: Brand New Day",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-07-31",
        "studio_distributor": "Sony Pictures / Marvel Studios",
        "director": "Destin Daniel Cretton",
        "source_name": "Sony Pictures"
    },

    {
        "title": "Michael",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-04-24",
        "studio_distributor": "Lionsgate",
        "director": "Antoine Fuqua",
        "source_name": "Lionsgate / Variety"
    },

    {
        "title": "The Odyssey",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-07-17",
        "studio_distributor": "Universal Pictures",
        "director": "Christopher Nolan",
        "source_name": "Universal / Variety"
    },

    {
        "title": "Scary Movie",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-06-05",
        "studio_distributor": "Paramount Pictures / Miramax",
        "director": "Michael Tiddes",
        "source_name": "Paramount"
    },

    {
        "title": "Mortal Kombat II",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-05-08",
        "studio_distributor": "Warner Bros. Pictures / New Line Cinema",
        "director": "Simon McQuoid",
        "source_name": "Warner Bros."
    },

    {
        "title": "The Super Mario Galaxy Movie",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-04-03",
        "studio_distributor": "Universal Pictures / Illumination / Nintendo",
        "director": "Aaron Horvath; Michael Jelenic; Pierre Leduc",
        "source_name": "Universal / Variety"
    },

    {
        "title": "The Devil Wears Prada 2",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-05-01",
        "studio_distributor": "20th Century Studios",
        "director": "David Frankel",
        "source_name": "20th Century Studios"
    },

    {
        "title": "Toy Story 5",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-06-19",
        "studio_distributor": "Disney / Pixar",
        "director": "Andrew Stanton",
        "source_name": "Pixar / Disney"
    },

    {
        "title": "GOAT",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-02-13",
        "studio_distributor": "Columbia Pictures / Sony Pictures Animation",
        "director": "Tyree Dillihay",
        "source_name": "Sony Pictures"
    },

    {
        "title": "Backrooms",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-05-29",
        "studio_distributor": "A24",
        "director": "Kane Parsons",
        "source_name": "A24"
    },

    {
        "title": "The Cat in the Hat",
        "cohort": "released_backtest",
        "release_status": "released",
        "release_date": "2026-02-26",
        "studio_distributor": "Warner Bros. Pictures Animation",
        "director": "Alessandro Carloni; Erica Rivinoja",
        "source_name": "Warner Bros. / Variety"
    },

    # FUTURE PREDICTIONS

    {
        "title": "Dune: Part Three",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2026-12-18",
        "studio_distributor": "Warner Bros. Pictures / Legendary",
        "director": "Denis Villeneuve",
        "source_name": "Warner Bros. / Variety"
    },

    {
        "title": "Avengers: Doomsday",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2026-12-18",
        "studio_distributor": "Marvel Studios / Disney",
        "director": "Anthony Russo; Joe Russo",
        "source_name": "Disney"
    },

    {
        "title": "Shrek 5",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2027-06-30",
        "studio_distributor": "DreamWorks Animation / Universal Pictures",
        "director": "Conrad Vernon; Walt Dohrn",
        "source_name": "DreamWorks / Variety"
    },

    {
        "title": "Jumanji: Open World",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2026-12-25",
        "studio_distributor": "Sony Pictures / Columbia Pictures",
        "director": "Jake Kasdan",
        "source_name": "Sony / Variety"
    },

    {
        "title": "Street Fighter",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2026-10-16",
        "studio_distributor": "Paramount Pictures / Legendary / Capcom",
        "director": "Kitao Sakurai",
        "source_name": "Paramount / Variety"
    },

    {
        "title": "The Batman Part II",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2028-02-18",
        "studio_distributor": "Warner Bros. Pictures / DC Studios",
        "director": "Matt Reeves",
        "source_name": "Warner Bros. / Variety"
    },

    {
        "title": "Spider-Man: Beyond the Spider-Verse",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2027-06-18",
        "studio_distributor": "Sony Pictures Animation",
        "director": "Bob Persichetti; Justin K. Thompson",
        "source_name": "Sony / Variety"
    },

    {
        "title": "Children of Blood and Bone",
        "cohort": "future_prediction",
        "release_status": "upcoming",
        "release_date": "2027-01-15",
        "studio_distributor": "Paramount Pictures",
        "director": "Gina Prince-Bythewood",
        "source_name": "Paramount"
    }
])


verified_targets["release_date"] = pd.to_datetime(
    verified_targets["release_date"]
)

verified_targets["release_year"] = (
    verified_targets["release_date"].dt.year
)


display(verified_targets)

,title,cohort,release_status,release_date,studio_distributor,director,source_name,release_year
0,Spider-Man: Brand New Day,released_backtest,released,2026-07-31,Sony Pictures / Marvel Studios,Destin Daniel Cretton,Sony Pictures,2026
1,Michael,released_backtest,released,2026-04-24,Lionsgate,Antoine Fuqua,Lionsgate / Variety,2026
2,The Odyssey,released_backtest,released,2026-07-17,Universal Pictures,Christopher Nolan,Universal / Variety,2026
3,Scary Movie,released_backtest,released,2026-06-05,Paramount Pictures / Miramax,Michael Tiddes,Paramount,2026
4,Mortal Kombat II,released_backtest,released,2026-05-08,Warner Bros. Pictures / New Line Cinema,Simon McQuoid,Warner Bros.,2026
5,The Super Mario Galaxy Movie,released_backtest,released,2026-04-03,Universal Pictures / Illumination / Nintendo,Aaron Horvath; Michael Jelenic; Pierre Leduc,Universal / Variety,2026
6,The Devil Wears Prada 2,released_backtest,released,2026-05-01,20th Century Studios,David Frankel,20th Century Studios,2026
7,Toy Story 5,released_backtest,released,2026-06-19,Disney / Pixar,Andrew Stanton,Pixar / Disney,2026
8,GOAT,released_backtest,released,2026-02-13,Columbia Pictures / Sony Pictures Animation,Tyree Dillihay,Sony Pictures,2026
9,Backrooms,released_backtest,released,2026-05-29,A24,Kane Parsons,A24,2026


## 06 ADD PREDICTION FEATURE COLLECTION COLUMNS

In [8]:
target_data = verified_targets.copy()


new_columns = {

    # Core characteristics
    "budget": np.nan,
    "budget_type": pd.NA,

    "genre": pd.NA,

    "production_company": pd.NA,

    "lead_cast": pd.NA,

    # Franchise
    "is_franchise": pd.NA,
    "is_sequel": pd.NA,
    "franchise_name": pd.NA,
    "franchise_gap_years": np.nan,

    # Release timing
    "release_month": np.nan,
    "release_season": pd.NA,
    "holiday_release": pd.NA,

    # Pre-release signals
    "pre_release_sentiment": np.nan,
    "social_interest": np.nan,

    # Historical performance
    "director_track_record": np.nan,
    "cast_track_record": np.nan,

    # Released cohort outcome
    "worldwide_box_office": np.nan,

    # Data quality
    "data_as_of": "2026-09-08",
    "collection_notes": pd.NA
}


for column, default_value in new_columns.items():

    target_data[column] = default_value


display(target_data.head())

,title,cohort,release_status,release_date,studio_distributor,director,source_name,release_year,budget,budget_type,...,release_month,release_season,holiday_release,pre_release_sentiment,social_interest,director_track_record,cast_track_record,worldwide_box_office,data_as_of,collection_notes
0,Spider-Man: Brand New Day,released_backtest,released,2026-07-31,Sony Pictures / Marvel Studios,Destin Daniel Cretton,Sony Pictures,2026,NaN,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,2026-09-08,<NA>
1,Michael,released_backtest,released,2026-04-24,Lionsgate,Antoine Fuqua,Lionsgate / Variety,2026,NaN,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,2026-09-08,<NA>
2,The Odyssey,released_backtest,released,2026-07-17,Universal Pictures,Christopher Nolan,Universal / Variety,2026,NaN,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,2026-09-08,<NA>
3,Scary Movie,released_backtest,released,2026-06-05,Paramount Pictures / Miramax,Michael Tiddes,Paramount,2026,NaN,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,2026-09-08,<NA>
4,Mortal Kombat II,released_backtest,released,2026-05-08,Warner Bros. Pictures / New Line Cinema,Simon McQuoid,Warner Bros.,2026,NaN,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,2026-09-08,<NA>


## 07 DERIVING RELEASE TIME

In [9]:
target_data["release_month"] = (
    target_data["release_date"]
    .dt.month
)


def assign_release_season(month):

    if month in [12, 1, 2]:
        return "summer_southern"

    elif month in [3, 4, 5]:
        return "autumn_southern"

    elif month in [6, 7, 8]:
        return "winter_southern"

    elif month in [9, 10, 11]:
        return "spring_southern"

    return pd.NA


target_data["release_season"] = (
    target_data["release_month"]
    .apply(assign_release_season)
)


display(
    target_data[
        [
            "title",
            "release_date",
            "release_year",
            "release_month",
            "release_season"
        ]
    ]
)

,title,release_date,release_year,release_month,release_season
0,Spider-Man: Brand New Day,2026-07-31,2026,7,winter_southern
1,Michael,2026-04-24,2026,4,autumn_southern
2,The Odyssey,2026-07-17,2026,7,winter_southern
3,Scary Movie,2026-06-05,2026,6,winter_southern
4,Mortal Kombat II,2026-05-08,2026,5,autumn_southern
5,The Super Mario Galaxy Movie,2026-04-03,2026,4,autumn_southern
6,The Devil Wears Prada 2,2026-05-01,2026,5,autumn_southern
7,Toy Story 5,2026-06-19,2026,6,winter_southern
8,GOAT,2026-02-13,2026,2,summer_southern
9,Backrooms,2026-05-29,2026,5,autumn_southern


## 08 VALIDATE PREDICTION REGISTER

In [10]:
print("=" * 80)
print("PREDICTION TARGET VALIDATION")
print("=" * 80)


print(
    f"Total targets: "
    f"{len(target_data)}"
)

print(
    f"Unique titles: "
    f"{target_data['title'].nunique()}"
)

print(
    f"Missing release dates: "
    f"{target_data['release_date'].isna().sum()}"
)

print(
    f"Missing directors: "
    f"{target_data['director'].isna().sum()}"
)


print("\nCOHORTS")
print("-" * 40)

print(
    target_data[
        "cohort"
    ].value_counts()
)


print("\nRELEASE STATUS")
print("-" * 40)

print(
    target_data[
        "release_status"
    ].value_counts()
)

PREDICTION TARGET VALIDATION
Total targets: 19
Unique titles: 19
Missing release dates: 0
Missing directors: 0

COHORTS
----------------------------------------
cohort
released_backtest    11
future_prediction     8
Name: count, dtype: int64

RELEASE STATUS
----------------------------------------
release_status
released    11
upcoming     8
Name: count, dtype: int64


## 09 SAVE INITIAL TARGET REGISTER

In [11]:
TARGET_DATA_PATH = (
    TARGET_DIR
    / "prediction_targets.csv"
)


target_data.to_csv(
    TARGET_DATA_PATH,
    index=False
)


print("=" * 80)
print("PREDICTION TARGET DATA SAVED")
print("=" * 80)

print(TARGET_DATA_PATH)

PREDICTION TARGET DATA SAVED
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets\prediction_targets.csv


## 10 BUDGET COLLECTION

In [13]:
# CORRECTING THE CAT IN THE HAT STATUS
cat_mask = (
    target_data["title"]
    .eq("The Cat in the Hat")
)

target_data.loc[
    cat_mask,
    "cohort"
] = "future_prediction"

target_data.loc[
    cat_mask,
    "release_status"
] = "upcoming"

target_data.loc[
    cat_mask,
    "release_date"
] = pd.Timestamp("2026-11-06")


# Recalculate date-derived fields
target_data["release_year"] = (
    target_data["release_date"]
    .dt.year
)

target_data["release_month"] = (
    target_data["release_date"]
    .dt.month
)

target_data["release_season"] = (
    target_data["release_month"]
    .apply(assign_release_season)
)


# BUDGET PROVENANCE COLUMNS

target_data["budget"] = np.nan
target_data["budget_type"] = pd.NA
target_data["budget_source"] = pd.NA
target_data["budget_notes"] = pd.NA


# VERIFIED / REPORTED BUDGETS
# Values in USD

budget_data = {

    "Spider-Man: Brand New Day": {
        "budget": 225_000_000,
        "type": "reported",
        "source": "Los Angeles Times / The Numbers",
        "notes": "Production budget reported at approximately $225M."
    },

    "Michael": {
        "budget": 170_000_000,
        "type": "reported_minimum",
        "source": "Variety",
        "notes": (
            "Originally greenlit around $155M; "
            "production changes and reshoots pushed "
            "reported cost to at least $170M."
        )
    },

    "The Odyssey": {
        "budget": 250_000_000,
        "type": "reported",
        "source": "Los Angeles Times / The Numbers",
        "notes": "Reported production budget of approximately $250M."
    },

    "Scary Movie": {
        "budget": 30_000_000,
        "type": "reported",
        "source": "The Numbers / Forbes",
        "notes": "Reported production budget of $30M."
    },

    "Mortal Kombat II": {
        "budget": 80_000_000,
        "type": "reported",
        "source": "The Numbers",
        "notes": "Reported production budget of $80M."
    },

    "The Super Mario Galaxy Movie": {
        "budget": 110_000_000,
        "type": "reported",
        "source": "Variety / The Numbers",
        "notes": "Reported production budget of $110M."
    },

    "The Devil Wears Prada 2": {
        "budget": 100_000_000,
        "type": "reported",
        "source": "Associated Press / The Numbers",
        "notes": "Reported production budget of $100M."
    },

    "Toy Story 5": {
        "budget": 175_000_000,
        "type": "range_midpoint_provisional",
        "source": "Los Angeles Times",
        "notes": (
            "Pre-release reporting placed production "
            "budget around $150M-$200M. "
            "$175M midpoint used provisionally."
        )
    },

    "GOAT": {
        "budget": 80_000_000,
        "type": "reported",
        "source": "Variety / The Numbers",
        "notes": "Reported production budget of $80M."
    },

    "Backrooms": {
        "budget": 10_000_000,
        "type": "reported",
        "source": "The Numbers",
        "notes": "Reported production budget of $10M."
    }
}


# APPLY BUDGET DATA

for title, info in budget_data.items():

    mask = (
        target_data["title"]
        .eq(title)
    )

    target_data.loc[
        mask,
        "budget"
    ] = info["budget"]

    target_data.loc[
        mask,
        "budget_type"
    ] = info["type"]

    target_data.loc[
        mask,
        "budget_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "budget_notes"
    ] = info["notes"]


# MARK CURRENTLY UNKNOWN FUTURE BUDGETS

unknown_budget_mask = (
    target_data["budget"].isna()
)

target_data.loc[
    unknown_budget_mask,
    "budget_type"
] = "unknown"


# DISPLAY RESULTS

print("=" * 80)
print("PREDICTION TARGET BUDGETS")
print("=" * 80)

print(
    f"Budget available: "
    f"{target_data['budget'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Budget unknown: "
    f"{target_data['budget'].isna().sum()}"
)

display(
    target_data[
        [
            "title",
            "cohort",
            "budget",
            "budget_type",
            "budget_source"
        ]
    ]
)

PREDICTION TARGET BUDGETS
Budget available: 10 of 19
Budget unknown: 9


,title,cohort,budget,budget_type,budget_source
0,Spider-Man: Brand New Day,released_backtest,225000000.0,reported,Los Angeles Times / The Numbers
1,Michael,released_backtest,170000000.0,reported_minimum,Variety
2,The Odyssey,released_backtest,250000000.0,reported,Los Angeles Times / The Numbers
3,Scary Movie,released_backtest,30000000.0,reported,The Numbers / Forbes
4,Mortal Kombat II,released_backtest,80000000.0,reported,The Numbers
5,The Super Mario Galaxy Movie,released_backtest,110000000.0,reported,Variety / The Numbers
6,The Devil Wears Prada 2,released_backtest,100000000.0,reported,Associated Press / The Numbers
7,Toy Story 5,released_backtest,175000000.0,range_midpoint_provisional,Los Angeles Times
8,GOAT,released_backtest,80000000.0,reported,Variety / The Numbers
9,Backrooms,released_backtest,10000000.0,reported,The Numbers


## 11 GENRE AND PRODUCTION COMPANY

In [ ]:
# Source-tracking columns
target_data["genre_source"] = pd.NA
target_data["production_company_source"] = pd.NA


movie_metadata = {

    "Spider-Man: Brand New Day": {
        "genre": "Action, Adventure, Fantasy",
        "production_company": (
            "Columbia Pictures; Pascal Pictures; Marvel Studios"
        ),
        "genre_source": "Sony Pictures / Rotten Tomatoes",
        "production_source": "Sony Pictures / Rotten Tomatoes"
    },

    "Michael": {
        "genre": "Biography, Drama, Music",
        "production_company": "GK Films",
        "genre_source": "STARZ / Rotten Tomatoes",
        "production_source": "Rotten Tomatoes / IMDb Company Credits"
    },

    "The Odyssey": {
        "genre": "Adventure, Action, Fantasy",
        "production_company": "Syncopy; Universal Pictures",
        "genre_source": "Universal Pictures / Rotten Tomatoes",
        "production_source": "Universal Pictures / Rotten Tomatoes"
    },

    "Scary Movie": {
        "genre": "Comedy, Horror",
        "production_company": (
            "Miramax; Wayans Bros. Entertainment"
        ),
        "genre_source": "Paramount Pictures",
        "production_source": "Rotten Tomatoes / IMDb Company Credits"
    },

    "Mortal Kombat II": {
        "genre": "Action, Adventure, Fantasy",
        "production_company": (
            "New Line Cinema; Atomic Monster; "
            "Broken Road Productions; Fireside Films"
        ),
        "genre_source": "Rotten Tomatoes",
        "production_source": "Warner Bros. Official Movie Site"
    },

    "The Super Mario Galaxy Movie": {
        "genre": (
            "Animation, Adventure, Comedy, Action"
        ),
        "production_company": (
            "Illumination Entertainment; Nintendo; Universal Pictures"
        ),
        "genre_source": "Nintendo / Illumination / Rotten Tomatoes",
        "production_source": "Nintendo / Illumination"
    },

    "The Devil Wears Prada 2": {
        "genre": "Comedy, Drama",
        "production_company": "Wendy Finerman Productions",
        "genre_source": "20th Century Studios",
        "production_source": "20th Century Studios / Rotten Tomatoes"
    },

    "Toy Story 5": {
        "genre": (
            "Animation, Adventure, Comedy, Kids & Family"
        ),
        "production_company": "Pixar Animation Studios",
        "genre_source": "Disney / Pixar",
        "production_source": "Pixar / Rotten Tomatoes"
    },

    "GOAT": {
        "genre": (
            "Animation, Comedy, Adventure, Sports"
        ),
        "production_company": (
            "Sony Pictures Animation; Unanimous Media"
        ),
        "genre_source": "Sony Pictures",
        "production_source": "Sony Pictures / Unanimous Media"
    },

    "Backrooms": {
        "genre": (
            "Horror, Sci-Fi, Fantasy, Mystery & Thriller"
        ),
        "production_company": (
            "A24; Chernin Entertainment; "
            "21 Laps Entertainment; Atomic Monster"
        ),
        "genre_source": "Rotten Tomatoes / A24",
        "production_source": "A24 / Rotten Tomatoes"
    },

    "The Cat in the Hat": {
        "genre": (
            "Animation, Comedy, Adventure, Fantasy, Kids & Family"
        ),
        "production_company": "Warner Bros. Animation",
        "genre_source": "Rotten Tomatoes / Warner Bros.",
        "production_source": "Rotten Tomatoes / Warner Bros."
    },

    "Dune: Part Three": {
        "genre": "Sci-Fi, Adventure, Action, Fantasy",
        "production_company": "Legendary Pictures",
        "genre_source": "Rotten Tomatoes",
        "production_source": "Warner Bros. / Legendary"
    },

    "Avengers: Doomsday": {
        "genre": (
            "Action, Adventure, Fantasy, Sci-Fi"
        ),
        "production_company": "Marvel Studios; AGBO",
        "genre_source": "Disney",
        "production_source": "Marvel / Rotten Tomatoes"
    },

    "Shrek 5": {
        "genre": (
            "Animation, Comedy, Adventure, Fantasy, Kids & Family"
        ),
        "production_company": "DreamWorks Animation",
        "genre_source": "DreamWorks / Rotten Tomatoes",
        "production_source": "DreamWorks Animation"
    },

    "Jumanji: Open World": {
        "genre": (
            "Adventure, Action, Comedy, Fantasy"
        ),
        "production_company": (
            "Columbia Pictures; Matt Tolmach Productions; "
            "Seven Bucks Productions; The Detective Agency"
        ),
        "genre_source": "Rotten Tomatoes",
        "production_source": "Rotten Tomatoes"
    },

    "Street Fighter": {
        "genre": "Action, Adventure",
        "production_company": (
            "Legendary Pictures; Capcom"
        ),
        "genre_source": "Legendary / Rotten Tomatoes",
        "production_source": "Legendary / Paramount"
    },

    "The Batman Part II": {
        "genre": (
            "Action, Adventure, Crime, Drama"
        ),
        "production_company": (
            "DC Entertainment; 6th & Idaho Productions"
        ),
        "genre_source": "Rotten Tomatoes",
        "production_source": "Rotten Tomatoes"
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "genre": (
            "Animation, Action, Adventure, Comedy, Fantasy"
        ),
        "production_company": (
            "Sony Pictures Animation; Arad Productions; "
            "Lord Miller; Pascal Pictures"
        ),
        "genre_source": "Sony Pictures Animation / Rotten Tomatoes",
        "production_source": "Sony Pictures Animation / Rotten Tomatoes"
    },

    "Children of Blood and Bone": {
        "genre": (
            "Action, Adventure, Drama, Fantasy"
        ),
        "production_company": (
            "Temple Hill Entertainment; Sunswept Entertainment"
        ),
        "genre_source": "Paramount Pictures / Rotten Tomatoes",
        "production_source": "The Numbers / IMDb Company Credits"
    }
}


# APPLY METADATA

for title, info in movie_metadata.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "genre"
    ] = info["genre"]

    target_data.loc[
        mask,
        "production_company"
    ] = info["production_company"]

    target_data.loc[
        mask,
        "genre_source"
    ] = info["genre_source"]

    target_data.loc[
        mask,
        "production_company_source"
    ] = info["production_source"]


# VALIDATION

print("=" * 80)
print("GENRE & PRODUCTION COMPANY COLLECTION")
print("=" * 80)

print(
    f"Genres available: "
    f"{target_data['genre'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Production companies available: "
    f"{target_data['production_company'].notna().sum()} "
    f"of {len(target_data)}"
)


display(
    target_data[
        [
            "title",
            "genre",
            "production_company",
            "genre_source",
            "production_company_source"
        ]
    ]
)

GENRE & PRODUCTION COMPANY COLLECTION
Genres available: 19 of 19
Production companies available: 19 of 19


,title,genre,production_company,genre_source,production_company_source
0,Spider-Man: Brand New Day,"Action, Adventure, Fantasy",Columbia Pictures; Pascal Pictures; Marvel Stu...,Sony Pictures / Rotten Tomatoes,Sony Pictures / Rotten Tomatoes
1,Michael,"Biography, Drama, Music",GK Films,STARZ / Rotten Tomatoes,Rotten Tomatoes / IMDb Company Credits
2,The Odyssey,"Adventure, Action, Fantasy",Syncopy; Universal Pictures,Universal Pictures / Rotten Tomatoes,Universal Pictures / Rotten Tomatoes
3,Scary Movie,"Comedy, Horror",Miramax; Wayans Bros. Entertainment,Paramount Pictures,Rotten Tomatoes / IMDb Company Credits
4,Mortal Kombat II,"Action, Adventure, Fantasy",New Line Cinema; Atomic Monster; Broken Road P...,Rotten Tomatoes,Warner Bros. Official Movie Site
5,The Super Mario Galaxy Movie,"Animation, Adventure, Comedy, Action",Illumination Entertainment; Nintendo; Universa...,Nintendo / Illumination / Rotten Tomatoes,Nintendo / Illumination
6,The Devil Wears Prada 2,"Comedy, Drama",Wendy Finerman Productions,20th Century Studios,20th Century Studios / Rotten Tomatoes
7,Toy Story 5,"Animation, Adventure, Comedy, Kids & Family",Pixar Animation Studios,Disney / Pixar,Pixar / Rotten Tomatoes
8,GOAT,"Animation, Comedy, Adventure, Sports",Sony Pictures Animation; Unanimous Media,Sony Pictures,Sony Pictures / Unanimous Media
9,Backrooms,"Horror, Sci-Fi, Fantasy, Mystery & Thriller",A24; Chernin Entertainment; 21 Laps Entertainm...,Rotten Tomatoes / A24,A24 / Rotten Tomatoes


## 12 LEAD CAST COLLECTION

In [16]:
# SOURCE-TRACKING COLUMNS

target_data["lead_cast_source"] = pd.NA
target_data["lead_cast_source_type"] = pd.NA


# LEAD CAST DATA
#
# Rule:
# Store up to four principal / prominently marketed cast
# members for each movie.

lead_cast_data = {

    "Spider-Man: Brand New Day": {
        "cast": (
            "Tom Holland; Zendaya; "
            "Sadie Sink; Jon Bernthal"
        ),
        "source": "Sony Pictures - Spider-Man: Brand New Day",
        "source_type": "official_studio"
    },

    "Michael": {
        "cast": (
            "Jaafar Jackson; Colman Domingo; "
            "Nia Long; Miles Teller"
        ),
        "source": "STARZ / Lionsgate",
        "source_type": "official_distributor"
    },

    "The Odyssey": {
        "cast": (
            "Matt Damon; Tom Holland; "
            "Anne Hathaway; Robert Pattinson"
        ),
        "source": "Universal Pictures",
        "source_type": "official_studio"
    },

    "Scary Movie": {
        "cast": (
            "Anna Faris; Regina Hall; "
            "Marlon Wayans; Shawn Wayans"
        ),
        "source": "Paramount Pictures / Paramount+",
        "source_type": "official_studio"
    },

    "Mortal Kombat II": {
        "cast": (
            "Karl Urban; Adeline Rudolph; "
            "Jessica McNamee; Lewis Tan"
        ),
        "source": "Warner Bros. Pictures",
        "source_type": "official_studio"
    },

    "The Super Mario Galaxy Movie": {
        "cast": (
            "Chris Pratt; Anya Taylor-Joy; "
            "Charlie Day; Jack Black"
        ),
        "source": "Nintendo / Illumination",
        "source_type": "official_studio"
    },

    "The Devil Wears Prada 2": {
        "cast": (
            "Meryl Streep; Anne Hathaway; "
            "Emily Blunt; Stanley Tucci"
        ),
        "source": "20th Century Studios",
        "source_type": "official_studio"
    },

    "Toy Story 5": {
        "cast": (
            "Tom Hanks; Tim Allen; "
            "Joan Cusack; Greta Lee"
        ),
        "source": "Disney / Pixar",
        "source_type": "official_studio"
    },

    "GOAT": {
        "cast": (
            "Caleb McLaughlin; Gabrielle Union; "
            "Stephen Curry; Aaron Pierre"
        ),
        "source": "Sony Pictures Animation / Rotten Tomatoes",
        "source_type": "studio_and_secondary"
    },

    "Backrooms": {
        "cast": (
            "Chiwetel Ejiofor; Renate Reinsve; "
            "Mark Duplass; Finn Bennett"
        ),
        "source": "A24",
        "source_type": "official_studio"
    },

    "The Cat in the Hat": {
        "cast": (
            "Bill Hader; Quinta Brunson; "
            "Xochitl Gomez; America Ferrera"
        ),
        "source": "Warner Bros. Pictures",
        "source_type": "official_studio"
    },

    "Dune: Part Three": {
        "cast": (
            "Timothee Chalamet; Zendaya; "
            "Jason Momoa; Florence Pugh"
        ),
        "source": "Warner Bros. / Rotten Tomatoes",
        "source_type": "studio_and_secondary"
    },

    "Avengers: Doomsday": {
        "cast": (
            "Robert Downey Jr.; Chris Evans; "
            "Chris Hemsworth; Pedro Pascal"
        ),
        "source": "Disney / Marvel Studios",
        "source_type": "official_studio"
    },

    "Shrek 5": {
        "cast": (
            "Mike Myers; Eddie Murphy; "
            "Cameron Diaz; Zendaya"
        ),
        "source": "Universal / DreamWorks Animation",
        "source_type": "official_studio"
    },

    "Jumanji: Open World": {
        "cast": (
            "Dwayne Johnson; Kevin Hart; "
            "Jack Black; Karen Gillan"
        ),
        "source": "Sony Pictures",
        "source_type": "official_studio"
    },

    "Street Fighter": {
        "cast": (
            "Andrew Koji; Noah Centineo; "
            "Callina Liang; Jason Momoa"
        ),
        "source": "Paramount Pictures / Legendary",
        "source_type": "official_studio"
    },

    "The Batman Part II": {
        "cast": (
            "Robert Pattinson; Scarlett Johansson; "
            "Sebastian Stan; Jeffrey Wright"
        ),
        "source": "Warner Bros. / Variety / Entertainment Weekly",
        "source_type": "studio_and_trade"
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "cast": (
            "Shameik Moore; Hailee Steinfeld; "
            "Brian Tyree Henry; Nicolas Cage"
        ),
        "source": "Sony Pictures Animation / IMDb",
        "source_type": "studio_and_secondary"
    },

    "Children of Blood and Bone": {
        "cast": (
            "Thuso Mbedu; Damson Idris; "
            "Amandla Stenberg; Tosin Cole"
        ),
        "source": "Paramount Pictures",
        "source_type": "official_studio"
    }
}


# APPLY CAST DATA

for title, info in lead_cast_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "lead_cast"
    ] = info["cast"]

    target_data.loc[
        mask,
        "lead_cast_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "lead_cast_source_type"
    ] = info["source_type"]


# VALIDATION

print("=" * 80)
print("LEAD CAST COLLECTION")
print("=" * 80)

print(
    f"Lead cast available: "
    f"{target_data['lead_cast'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Missing lead cast: "
    f"{target_data['lead_cast'].isna().sum()}"
)


print("\nSOURCE TYPES")
print("-" * 40)

print(
    target_data[
        "lead_cast_source_type"
    ].value_counts(
        dropna=False
    )
)


display(
    target_data[
        [
            "title",
            "lead_cast",
            "lead_cast_source",
            "lead_cast_source_type"
        ]
    ]
)

LEAD CAST COLLECTION
Lead cast available: 19 of 19
Missing lead cast: 0

SOURCE TYPES
----------------------------------------
lead_cast_source_type
official_studio         14
studio_and_secondary     3
official_distributor     1
studio_and_trade         1
Name: count, dtype: int64


,title,lead_cast,lead_cast_source,lead_cast_source_type
0,Spider-Man: Brand New Day,Tom Holland; Zendaya; Sadie Sink; Jon Bernthal,Sony Pictures - Spider-Man: Brand New Day,official_studio
1,Michael,Jaafar Jackson; Colman Domingo; Nia Long; Mile...,STARZ / Lionsgate,official_distributor
2,The Odyssey,Matt Damon; Tom Holland; Anne Hathaway; Robert...,Universal Pictures,official_studio
3,Scary Movie,Anna Faris; Regina Hall; Marlon Wayans; Shawn ...,Paramount Pictures / Paramount+,official_studio
4,Mortal Kombat II,Karl Urban; Adeline Rudolph; Jessica McNamee; ...,Warner Bros. Pictures,official_studio
5,The Super Mario Galaxy Movie,Chris Pratt; Anya Taylor-Joy; Charlie Day; Jac...,Nintendo / Illumination,official_studio
6,The Devil Wears Prada 2,Meryl Streep; Anne Hathaway; Emily Blunt; Stan...,20th Century Studios,official_studio
7,Toy Story 5,Tom Hanks; Tim Allen; Joan Cusack; Greta Lee,Disney / Pixar,official_studio
8,GOAT,Caleb McLaughlin; Gabrielle Union; Stephen Cur...,Sony Pictures Animation / Rotten Tomatoes,studio_and_secondary
9,Backrooms,Chiwetel Ejiofor; Renate Reinsve; Mark Duplass...,A24,official_studio


## 13 FRANCHISE AND SEQUEL INFORMATION

In [17]:
# SOURCE-TRACKING COLUMNS

target_data["existing_ip"] = pd.NA
target_data["franchise_relationship"] = pd.NA
target_data["franchise_source"] = pd.NA
target_data["franchise_source_url"] = pd.NA


# ------------------------------------------------------------
# FRANCHISE DATA
#
# Definitions:
#
# existing_ip:
#   Based on a recognisable pre-existing film, book, game,
#   comic, web series or entertainment property.
#
# is_franchise:
#   Part of an established entertainment franchise/series.
#
# is_sequel:
#   Continues a previous feature-film storyline or series.
#
# franchise_relationship:
#   original
#   sequel
#   reboot_or_new_adaptation
#   standalone_adaptation
# ------------------------------------------------------------

franchise_data = {

    "Spider-Man: Brand New Day": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Spider-Man",
        "relationship": "sequel",
        "source": "Sony Pictures / Marvel",
        "url": "https://www.sonypictures.com/movies/spidermanbrandnewday"
    },

    "Michael": {
        "existing_ip": False,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": pd.NA,
        "relationship": "original",
        "source": "Lionsgate",
        "url": pd.NA
    },

    "The Odyssey": {
        "existing_ip": True,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": pd.NA,
        "relationship": "standalone_adaptation",
        "source": "Universal Pictures",
        "url": pd.NA
    },

    "Scary Movie": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Scary Movie",
        "relationship": "sequel",
        "source": "Paramount Pictures",
        "url": "https://www.paramountpictures.com/movies/scary-movie"
    },

    "Mortal Kombat II": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Mortal Kombat",
        "relationship": "sequel",
        "source": "Warner Bros. / New Line Cinema",
        "url": pd.NA
    },

    "The Super Mario Galaxy Movie": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Super Mario",
        "relationship": "sequel",
        "source": "Nintendo / Illumination",
        "url": (
            "https://www.nintendo.co.jp/"
            "corporate/release/en/2025/250912.html"
        )
    },

    "The Devil Wears Prada 2": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "The Devil Wears Prada",
        "relationship": "sequel",
        "source": "20th Century Studios",
        "url": (
            "https://www.20thcenturystudios.com/"
            "movies/the-devil-wears-prada-2"
        )
    },

    "Toy Story 5": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Toy Story",
        "relationship": "sequel",
        "source": "Pixar",
        "url": "https://www.pixar.com/toy-story-5"
    },

    "GOAT": {
        "existing_ip": False,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": pd.NA,
        "relationship": "original",
        "source": "Sony Pictures",
        "url": "https://www.sonypictures.com/movies/goat"
    },

    "Backrooms": {
        "existing_ip": True,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": "Backrooms",
        "relationship": "standalone_adaptation",
        "source": "A24",
        "url": "https://a24films.com/films/backrooms"
    },

    "The Cat in the Hat": {
        "existing_ip": True,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": "Dr. Seuss - The Cat in the Hat",
        "relationship": "reboot_or_new_adaptation",
        "source": "Warner Bros. Pictures",
        "url": "https://www.catinthehatfilm.com/"
    },

    "Dune: Part Three": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Dune",
        "relationship": "sequel",
        "source": "Warner Bros. / Legendary",
        "url": "https://www.dunemovie.net/"
    },

    "Avengers: Doomsday": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Avengers / Marvel Cinematic Universe",
        "relationship": "sequel",
        "source": "Marvel / Disney",
        "url": "https://movies.disney.com/avengers-doomsday"
    },

    "Shrek 5": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Shrek",
        "relationship": "sequel",
        "source": "DreamWorks Animation",
        "url": "https://www.dreamworks.com/shrek"
    },

    "Jumanji: Open World": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Jumanji",
        "relationship": "sequel",
        "source": "Sony Pictures",
        "url": "https://www.sonypictures.com/"
    },

    "Street Fighter": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": False,
        "franchise_name": "Street Fighter",
        "relationship": "reboot_or_new_adaptation",
        "source": "Paramount / Legendary / Capcom",
        "url": (
            "https://www.paramount.com/press/"
            "paramount-pictures-and-legendary-entertainment-"
            "strike-strategic-three-year-global-distribution-deal"
        )
    },

    "The Batman Part II": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "The Batman",
        "relationship": "sequel",
        "source": "DC Studios / Warner Bros.",
        "url": (
            "https://www.dc.com/blog/2023/01/31/"
            "the-next-generation-of-dc-movies-and-tv-has-arrived"
        )
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "existing_ip": True,
        "is_franchise": True,
        "is_sequel": True,
        "franchise_name": "Spider-Verse",
        "relationship": "sequel",
        "source": "Sony Pictures Animation",
        "url": (
            "https://www.sonypicturesanimation.com/"
            "projects/films/spider-man-beyond-spider-verse"
        )
    },

    "Children of Blood and Bone": {
        "existing_ip": True,
        "is_franchise": False,
        "is_sequel": False,
        "franchise_name": "Legacy of Orisha",
        "relationship": "standalone_adaptation",
        "source": "Paramount Pictures",
        "url": (
            "https://www.paramountpictures.com/"
            "movies/children-of-blood-and-bone"
        )
    }
}

# APPLY FRANCHISE DATA

for title, info in franchise_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "existing_ip"
    ] = info["existing_ip"]

    target_data.loc[
        mask,
        "is_franchise"
    ] = info["is_franchise"]

    target_data.loc[
        mask,
        "is_sequel"
    ] = info["is_sequel"]

    target_data.loc[
        mask,
        "franchise_name"
    ] = info["franchise_name"]

    target_data.loc[
        mask,
        "franchise_relationship"
    ] = info["relationship"]

    target_data.loc[
        mask,
        "franchise_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "franchise_source_url"
    ] = info["url"]


# VALIDATION

print("=" * 80)
print("FRANCHISE & SEQUEL INFORMATION")
print("=" * 80)

print(
    f"Existing IP: "
    f"{target_data['existing_ip'].eq(True).sum()} "
    f"of {len(target_data)}"
)

print(
    f"Franchise movies: "
    f"{target_data['is_franchise'].eq(True).sum()} "
    f"of {len(target_data)}"
)

print(
    f"Sequels: "
    f"{target_data['is_sequel'].eq(True).sum()} "
    f"of {len(target_data)}"
)


print("\nFRANCHISE RELATIONSHIP")
print("-" * 40)

print(
    target_data[
        "franchise_relationship"
    ].value_counts()
)


display(
    target_data[
        [
            "title",
            "existing_ip",
            "is_franchise",
            "is_sequel",
            "franchise_name",
            "franchise_relationship",
            "franchise_source"
        ]
    ]
)

FRANCHISE & SEQUEL INFORMATION
Existing IP: 17 of 19
Franchise movies: 13 of 19
Sequels: 12 of 19

FRANCHISE RELATIONSHIP
----------------------------------------
franchise_relationship
sequel                      12
standalone_adaptation        3
original                     2
reboot_or_new_adaptation     2
Name: count, dtype: int64


,title,existing_ip,is_franchise,is_sequel,franchise_name,franchise_relationship,franchise_source
0,Spider-Man: Brand New Day,True,True,True,Spider-Man,sequel,Sony Pictures / Marvel
1,Michael,False,False,False,<NA>,original,Lionsgate
2,The Odyssey,True,False,False,<NA>,standalone_adaptation,Universal Pictures
3,Scary Movie,True,True,True,Scary Movie,sequel,Paramount Pictures
4,Mortal Kombat II,True,True,True,Mortal Kombat,sequel,Warner Bros. / New Line Cinema
5,The Super Mario Galaxy Movie,True,True,True,Super Mario,sequel,Nintendo / Illumination
6,The Devil Wears Prada 2,True,True,True,The Devil Wears Prada,sequel,20th Century Studios
7,Toy Story 5,True,True,True,Toy Story,sequel,Pixar
8,GOAT,False,False,False,<NA>,original,Sony Pictures
9,Backrooms,True,False,False,Backrooms,standalone_adaptation,A24


## 14 FRANCHISE GAP YEARS

In [19]:
# SOURCE-TRACKING COLUMNS

target_data["previous_franchise_film"] = pd.NA
target_data["previous_film_release_date"] = pd.NaT
target_data["franchise_gap_type"] = pd.NA
target_data["franchise_gap_source"] = pd.NA
target_data["franchise_gap_source_url"] = pd.NA


# PREVIOUS RELEVANT THEATRICAL FILMS
#
# gap_type:
#   direct_sequel
#       Previous film in the same continuing series.
#
#   legacy_sequel
#       Very long-gap continuation of an older series.
#
#   reboot_reference
#       Previous comparable theatrical adaptation, even
#       though the new movie is not a direct sequel.
#
# Movies with no relevant prior theatrical feature remain NA.

franchise_gap_data = {

    "Spider-Man: Brand New Day": {
        "previous_film": "Spider-Man: No Way Home",
        "previous_date": "2021-12-17",
        "gap_type": "direct_sequel",
        "source": "Sony Pictures",
        "url": (
            "https://www.sonypictures.com/"
            "movies/spidermannowayhome"
        )
    },

    "Scary Movie": {
        "previous_film": "Scary Movie 5",
        "previous_date": "2013-04-12",
        "gap_type": "legacy_sequel",
        "source": "Paramount / historical release records",
        "url": (
            "https://www.paramount.com/news/"
            "scary-movie-stars-are-ready-to-"
            "make-audiences-laugh-again"
        )
    },

    "Mortal Kombat II": {
        "previous_film": "Mortal Kombat",
        "previous_date": "2021-04-23",
        "gap_type": "direct_sequel",
        "source": "Warner Bros. / New Line Cinema",
        "url": pd.NA
    },

    "The Super Mario Galaxy Movie": {
        "previous_film": "The Super Mario Bros. Movie",
        "previous_date": "2023-04-05",
        "gap_type": "direct_sequel",
        "source": "Nintendo",
        "url": (
            "https://www.nintendo.com/en-ca/"
            "whatsnew/celebrate-the-launch-of-"
            "the-super-mario-bros-movie/"
        )
    },

    "The Devil Wears Prada 2": {
        "previous_film": "The Devil Wears Prada",
        "previous_date": "2006-06-30",
        "gap_type": "legacy_sequel",
        "source": "20th Century Studios",
        "url": (
            "https://www.20thcenturystudios.com/"
            "movies/the-devil-wears-prada"
        )
    },

    "Toy Story 5": {
        "previous_film": "Toy Story 4",
        "previous_date": "2019-06-21",
        "gap_type": "direct_sequel",
        "source": "Disney / Pixar",
        "url": (
            "https://movies.disney.com/"
            "toy-story-4"
        )
    },

    "The Cat in the Hat": {
        "previous_film": "The Cat in the Hat",
        "previous_date": "2003-11-21",
        "gap_type": "reboot_reference",
        "source": "Universal / historical release records",
        "url": pd.NA
    },

    "Dune: Part Three": {
        "previous_film": "Dune: Part Two",
        "previous_date": "2024-03-01",
        "gap_type": "direct_sequel",
        "source": "Warner Bros. / Legendary",
        "url": pd.NA
    },

    "Avengers: Doomsday": {
        "previous_film": "Avengers: Endgame",
        "previous_date": "2019-04-26",
        "gap_type": "direct_sequel",
        "source": "Disney / Marvel Studios",
        "url": (
            "https://movies.disney.com/"
            "avengers-endgame"
        )
    },

    "Shrek 5": {
        "previous_film": "Shrek Forever After",
        "previous_date": "2010-05-21",
        "gap_type": "legacy_sequel",
        "source": "DreamWorks Animation",
        "url": (
            "https://www.dreamworks.com/"
            "movies/shrek-forever-after"
        )
    },

    "Jumanji: Open World": {
        "previous_film": "Jumanji: The Next Level",
        "previous_date": "2019-12-13",
        "gap_type": "direct_sequel",
        "source": "Sony Pictures",
        "url": (
            "https://www.sonypictures.com/"
            "movies/jumanjithenextlevel"
        )
    },

    "Street Fighter": {
        "previous_film": (
            "Street Fighter: "
            "The Legend of Chun-Li"
        ),
        "previous_date": "2009-02-27",
        "gap_type": "reboot_reference",
        "source": "Capcom / historical release records",
        "url": (
            "https://news.capcomusa.com/"
            "lets/browse/street-fighter-"
            "comics-turning-into-dvds"
        )
    },

    "The Batman Part II": {
        "previous_film": "The Batman",
        "previous_date": "2022-03-04",
        "gap_type": "direct_sequel",
        "source": "DC / Warner Bros.",
        "url": (
            "https://www.dc.com/"
            "movies/the-batman-2022"
        )
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "previous_film": (
            "Spider-Man: "
            "Across the Spider-Verse"
        ),
        "previous_date": "2023-06-02",
        "gap_type": "direct_sequel",
        "source": "Sony Pictures",
        "url": (
            "https://www.sonypictures.com/"
            "movies/spidermanacrossthespiderverse"
        )
    }
}


# APPLY REFERENCE DATA

for title, info in franchise_gap_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "previous_franchise_film"
    ] = info["previous_film"]

    target_data.loc[
        mask,
        "previous_film_release_date"
    ] = pd.Timestamp(
        info["previous_date"]
    )

    target_data.loc[
        mask,
        "franchise_gap_type"
    ] = info["gap_type"]

    target_data.loc[
        mask,
        "franchise_gap_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "franchise_gap_source_url"
    ] = info["url"]


# CALCULATE GAP

target_data["franchise_gap_years"] = (
    (
        target_data["release_date"]
        - target_data["previous_film_release_date"]
    ).dt.days
    / 365.25
)


target_data["franchise_gap_years"] = (
    target_data[
        "franchise_gap_years"
    ].round(2)
)


# VALIDATION

print("=" * 80)
print("FRANCHISE / LEGACY GAP YEARS")
print("=" * 80)

print(
    f"Movies with calculated gap: "
    f"{target_data['franchise_gap_years'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Not applicable / no prior feature: "
    f"{target_data['franchise_gap_years'].isna().sum()}"
)


print("\nGAP TYPES")
print("-" * 40)

print(
    target_data[
        "franchise_gap_type"
    ].value_counts(
        dropna=False
    )
)


display(
    target_data[
        [
            "title",
            "previous_franchise_film",
            "previous_film_release_date",
            "release_date",
            "franchise_gap_years",
            "franchise_gap_type",
            "franchise_gap_source"
        ]
    ].sort_values(
        "franchise_gap_years",
        ascending=False,
        na_position="last"
    )
)

FRANCHISE / LEGACY GAP YEARS
Movies with calculated gap: 14 of 19
Not applicable / no prior feature: 5

GAP TYPES
----------------------------------------
franchise_gap_type
direct_sequel       9
<NA>                5
legacy_sequel       3
reboot_reference    2
Name: count, dtype: int64


,title,previous_franchise_film,previous_film_release_date,release_date,franchise_gap_years,franchise_gap_type,franchise_gap_source
10,The Cat in the Hat,The Cat in the Hat,2003-11-21,2026-11-06,22.96,reboot_reference,Universal / historical release records
6,The Devil Wears Prada 2,The Devil Wears Prada,2006-06-30,2026-05-01,19.84,legacy_sequel,20th Century Studios
15,Street Fighter,Street Fighter: The Legend of Chun-Li,2009-02-27,2026-10-16,17.63,reboot_reference,Capcom / historical release records
13,Shrek 5,Shrek Forever After,2010-05-21,2027-06-30,17.11,legacy_sequel,DreamWorks Animation
3,Scary Movie,Scary Movie 5,2013-04-12,2026-06-05,13.15,legacy_sequel,Paramount / historical release records
12,Avengers: Doomsday,Avengers: Endgame,2019-04-26,2026-12-18,7.65,direct_sequel,Disney / Marvel Studios
14,Jumanji: Open World,Jumanji: The Next Level,2019-12-13,2026-12-25,7.03,direct_sequel,Sony Pictures
7,Toy Story 5,Toy Story 4,2019-06-21,2026-06-19,7.00,direct_sequel,Disney / Pixar
16,The Batman Part II,The Batman,2022-03-04,2028-02-18,5.96,direct_sequel,DC / Warner Bros.
4,Mortal Kombat II,Mortal Kombat,2021-04-23,2026-05-08,5.04,direct_sequel,Warner Bros. / New Line Cinema


## 15 WORLDWIDE BOX OFFICE OUTCOMES

In [20]:
# SOURCE / STATUS COLUMNS

target_data["worldwide_box_office_source"] = pd.NA
target_data["box_office_as_of"] = pd.NaT
target_data["box_office_status"] = pd.NA


# WORLDWIDE BOX OFFICE
#
# Primary source:
# The Numbers
#
# Values are USD.
#
# IMPORTANT:
# Some 2026 films may still have limited theatrical activity,
# so these values represent the latest available total at the
# time of collection rather than permanently frozen figures.

box_office_data = {

    "Spider-Man: Brand New Day": {
        "worldwide": 2_402_937_000,
        "status": "provisional_active_run",
        "source": "The Numbers"
    },

    "Michael": {
        "worldwide": 1_021_426_595,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "The Odyssey": {
        "worldwide": 1_627_387_000,
        "status": "provisional_active_run",
        "source": "The Numbers"
    },

    "Scary Movie": {
        "worldwide": 231_277_762,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "Mortal Kombat II": {
        "worldwide": 129_470_110,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "The Super Mario Galaxy Movie": {
        "worldwide": 1_012_793_366,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "The Devil Wears Prada 2": {
        "worldwide": 692_817_491,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "Toy Story 5": {
        "worldwide": 1_137_319_581,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "GOAT": {
        "worldwide": 195_356_870,
        "status": "reported_total",
        "source": "The Numbers"
    },

    "Backrooms": {
        "worldwide": 393_167_286,
        "status": "reported_total",
        "source": "The Numbers"
    }
}


# APPLY BOX OFFICE DATA

for title, info in box_office_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "worldwide_box_office"
    ] = info["worldwide"]

    target_data.loc[
        mask,
        "worldwide_box_office_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "box_office_status"
    ] = info["status"]

    target_data.loc[
        mask,
        "box_office_as_of"
    ] = pd.Timestamp("2026-09-08")


# ENSURE FUTURE MOVIES HAVE NO OUTCOME

future_mask = (
    target_data["cohort"]
    .eq("future_prediction")
)

target_data.loc[
    future_mask,
    "worldwide_box_office"
] = np.nan


# VALIDATION

released_mask = (
    target_data["cohort"]
    .eq("released_backtest")
)


print("=" * 80)
print("RELEASED MOVIE WORLDWIDE BOX OFFICE")
print("=" * 80)

print(
    f"Released/backtest movies: "
    f"{released_mask.sum()}"
)

print(
    f"Worldwide outcomes available: "
    f"{target_data.loc[released_mask, 'worldwide_box_office'].notna().sum()}"
)

print(
    f"Missing released outcomes: "
    f"{target_data.loc[released_mask, 'worldwide_box_office'].isna().sum()}"
)


display(
    target_data.loc[
        released_mask,
        [
            "title",
            "budget",
            "worldwide_box_office",
            "box_office_status",
            "worldwide_box_office_source",
            "box_office_as_of"
        ]
    ].sort_values(
        "worldwide_box_office",
        ascending=False
    )
)

RELEASED MOVIE WORLDWIDE BOX OFFICE
Released/backtest movies: 10
Worldwide outcomes available: 10
Missing released outcomes: 0


,title,budget,worldwide_box_office,box_office_status,worldwide_box_office_source,box_office_as_of
0,Spider-Man: Brand New Day,225000000.0,2.402937e+09,provisional_active_run,The Numbers,2026-09-08
2,The Odyssey,250000000.0,1.627387e+09,provisional_active_run,The Numbers,2026-09-08
7,Toy Story 5,175000000.0,1.137320e+09,reported_total,The Numbers,2026-09-08
1,Michael,170000000.0,1.021427e+09,reported_total,The Numbers,2026-09-08
5,The Super Mario Galaxy Movie,110000000.0,1.012793e+09,reported_total,The Numbers,2026-09-08
6,The Devil Wears Prada 2,100000000.0,6.928175e+08,reported_total,The Numbers,2026-09-08
9,Backrooms,10000000.0,3.931673e+08,reported_total,The Numbers,2026-09-08
3,Scary Movie,30000000.0,2.312778e+08,reported_total,The Numbers,2026-09-08
8,GOAT,80000000.0,1.953569e+08,reported_total,The Numbers,2026-09-08
4,Mortal Kombat II,80000000.0,1.294701e+08,reported_total,The Numbers,2026-09-08


## 16 DIRECTOR AND CAST HISTORICAL TRACK RECORD

In [ ]:
HISTORICAL_DATA_PATH = (
    DATA_DIR
    / "Clean"
    / "cleaned_movies.csv"
)

historical_movies = pd.read_csv(
    HISTORICAL_DATA_PATH
)


print("=" * 80)
print("HISTORICAL TRACK-RECORD DATA")
print("=" * 80)

print(
    f"Historical movies loaded: "
    f"{len(historical_movies):,}"
)


# NAME NORMALIZATION

def normalize_person_name(name):
    """
    Normalize person names for matching.
    """

    if pd.isna(name):
        return pd.NA

    name = str(name).strip().lower()

    name = unicodedata.normalize(
        "NFKD",
        name
    )

    name = "".join(
        character
        for character in name
        if not unicodedata.combining(character)
    )

    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name if name else pd.NA


# PREPARE HISTORICAL DATA

historical_movies["_director_key"] = (
    historical_movies["director"]
    .apply(normalize_person_name)
)

historical_movies["_star_key"] = (
    historical_movies["star"]
    .apply(normalize_person_name)
)

historical_movies["_gross_numeric"] = (
    pd.to_numeric(
        historical_movies["gross"],
        errors="coerce"
    )
)

historical_movies["_year_numeric"] = (
    pd.to_numeric(
        historical_movies["year"],
        errors="coerce"
    )
)


# Only financially usable historical films
historical_financial = historical_movies[
    historical_movies["_gross_numeric"].gt(0)
].copy()


# TRACK RECORD FUNCTIONS

def director_track_record(
    director,
    target_year
):
    """
    Median historical gross for films directed
    by the target director before the target year.
    """

    director_key = normalize_person_name(
        director
    )

    if pd.isna(director_key):
        return np.nan, 0

    prior_films = historical_financial[
        historical_financial[
            "_director_key"
        ].eq(director_key)
        &
        historical_financial[
            "_year_numeric"
        ].lt(target_year)
    ]

    if prior_films.empty:
        return np.nan, 0

    return (
        float(
            prior_films[
                "_gross_numeric"
            ].median()
        ),
        int(len(prior_films))
    )


def cast_track_record(
    lead_cast,
    target_year
):
    #Median historical gross of films starring
    #any listed lead-cast member before target year.

    if pd.isna(lead_cast):
        return np.nan, 0, 0

    cast_members = [
        normalize_person_name(name)
        for name in str(
            lead_cast
        ).split(";")
    ]

    cast_members = [
        name
        for name in cast_members
        if pd.notna(name)
    ]

    if not cast_members:
        return np.nan, 0, 0

    prior_films = historical_financial[
        historical_financial[
            "_star_key"
        ].isin(cast_members)
        &
        historical_financial[
            "_year_numeric"
        ].lt(target_year)
    ]

    matched_cast_members = (
        prior_films[
            "_star_key"
        ].nunique()
    )

    if prior_films.empty:
        return np.nan, 0, 0

    return (
        float(
            prior_films[
                "_gross_numeric"
            ].median()
        ),
        int(len(prior_films)),
        int(matched_cast_members)
    )


# CALCULATE DIRECTOR FEATURES

director_results = target_data.apply(
    lambda row:
        director_track_record(
            row["director"],
            row["release_year"]
        ),
    axis=1
)


target_data["director_track_record"] = [
    result[0]
    for result in director_results
]

target_data["director_prior_films"] = [
    result[1]
    for result in director_results
]


# CALCULATE CAST FEATURES

cast_results = target_data.apply(
    lambda row:
        cast_track_record(
            row["lead_cast"],
            row["release_year"]
        ),
    axis=1
)


target_data["cast_track_record"] = [
    result[0]
    for result in cast_results
]

target_data["cast_prior_films"] = [
    result[1]
    for result in cast_results
]

target_data["matched_lead_cast_members"] = [
    result[2]
    for result in cast_results
]


# SOURCE / METHOD TRACKING

target_data[
    "director_track_record_source"
] = (
    "Dataset A - cleaned_movies.csv"
)

target_data[
    "cast_track_record_source"
] = (
    "Dataset A - cleaned_movies.csv"
)

target_data[
    "track_record_method"
] = (
    "Median historical gross of prior films"
)


# VALIDATION

print("\n" + "=" * 80)
print("DIRECTOR & CAST TRACK RECORD")
print("=" * 80)


print(
    f"Director track records available: "
    f"{target_data['director_track_record'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Cast track records available: "
    f"{target_data['cast_track_record'].notna().sum()} "
    f"of {len(target_data)}"
)


display(
    target_data[
        [
            "title",
            "director",
            "director_prior_films",
            "director_track_record",
            "lead_cast",
            "matched_lead_cast_members",
            "cast_prior_films",
            "cast_track_record"
        ]
    ]
)

HISTORICAL TRACK-RECORD DATA
Historical movies loaded: 7,668

DIRECTOR & CAST TRACK RECORD
Director track records available: 10 of 19
Cast track records available: 19 of 19


,title,director,director_prior_films,director_track_record,lead_cast,matched_lead_cast_members,cast_prior_films,cast_track_record
0,Spider-Man: Brand New Day,Destin Daniel Cretton,2,11866848.5,Tom Holland; Zendaya; Sadie Sink; Jon Bernthal,2,4,511058766.0
1,Michael,Antoine Fuqua,12,100286614.5,Jaafar Jackson; Colman Domingo; Nia Long; Mile...,1,6,27492131.5
2,The Odyssey,Christopher Nolan,11,373661946.0,Matt Damon; Tom Holland; Anne Hathaway; Robert...,4,45,100266865.0
3,Scary Movie,Michael Tiddes,3,25358716.0,Anna Faris; Regina Hall; Marlon Wayans; Shawn ...,4,18,65299640.0
4,Mortal Kombat II,Simon McQuoid,0,NaN,Karl Urban; Adeline Rudolph; Jessica McNamee; ...,1,4,36011162.5
5,The Super Mario Galaxy Movie,Aaron Horvath; Michael Jelenic; Pierre Leduc,0,NaN,Chris Pratt; Anya Taylor-Joy; Charlie Day; Jac...,3,20,175283966.0
6,The Devil Wears Prada 2,David Frankel,7,88528280.0,Meryl Streep; Anne Hathaway; Emily Blunt; Stan...,3,41,51699984.0
7,Toy Story 5,Andrew Stanton,4,730832267.5,Tom Hanks; Tim Allen; Joan Cusack; Greta Lee,2,51,132440069.0
8,GOAT,Tyree Dillihay,0,NaN,Caleb McLaughlin; Gabrielle Union; Stephen Cur...,1,1,31609243.0
9,Backrooms,Kane Parsons,0,NaN,Chiwetel Ejiofor; Renate Reinsve; Mark Duplass...,2,8,6596467.5


## 17 PRE-RELEASE SENTIMENT COLLECTION

In [ ]:
# ------------------------------------------------------------
# SENTIMENT SCALE
#
# +2 = strongly positive
# +1 = positive
#  0 = mixed / neutral
# -1 = negative
# -2 = strongly negative
#
# NaN = insufficient reliable pre-release evidence
# ------------------------------------------------------------

target_data["pre_release_sentiment_label"] = pd.NA
target_data["sentiment_confidence"] = pd.NA
target_data["sentiment_source"] = pd.NA
target_data["sentiment_source_url"] = pd.NA
target_data["sentiment_as_of"] = pd.NaT
target_data["sentiment_stage"] = pd.NA


sentiment_data = {

    # RELEASED / BACKTEST MOVIES
    # Evidence must pre-date theatrical release.

    "Spider-Man: Brand New Day": {
        "score": 2,
        "label": "strong_positive",
        "confidence": "high",
        "source": "Variety - First Reactions",
        "url": (
            "https://variety.com/2026/film/news/"
            "spider-man-brand-new-day-first-reactions-"
            "tom-holland-1236476079/"
        ),
        "date": "2026-07-28",
        "stage": "first_reactions"
    },

    "Michael": {
        "score": 0,
        "label": "mixed",
        "confidence": "high",
        "source": "Variety",
        "url": (
            "https://variety.com/2026/film/news/"
            "michael-box-office-record-opening-"
            "music-biopic-1236470847/"
        ),
        "date": "2026-04-23",
        "stage": "pre_release_press"
    },

    "The Odyssey": {
        "score": 1,
        "label": "positive",
        "confidence": "medium",
        "source": "Variety - World Premiere Coverage",
        "url": (
            "https://variety.com/2026/film/news/"
            "the-odyssey-world-premiere-london-1236454419/"
        ),
        "date": "2026-07-07",
        "stage": "premiere_buzz"
    },

    "Scary Movie": {
        "score": np.nan,
        "label": "insufficient_evidence",
        "confidence": "low",
        "source": pd.NA,
        "url": pd.NA,
        "date": pd.NaT,
        "stage": pd.NA
    },

    "Mortal Kombat II": {
        "score": -1,
        "label": "negative",
        "confidence": "high",
        "source": "Variety - Pre-Release Review",
        "url": (
            "https://variety.com/2026/film/news/"
            "mortal-kombat-ii-review-karl-urban-1236447591/"
        ),
        "date": "2026-05-07",
        "stage": "pre_release_review"
    },

    "The Super Mario Galaxy Movie": {
        "score": -1,
        "label": "negative",
        "confidence": "high",
        "source": "Variety - Pre-Release Review",
        "url": (
            "https://variety.com/2026/film/global/"
            "the-super-mario-galaxy-movie-review-"
            "chris-pratt-jack-black-1236418581/"
        ),
        "date": "2026-04-01",
        "stage": "pre_release_review"
    },

    "The Devil Wears Prada 2": {
        "score": 2,
        "label": "strong_positive",
        "confidence": "high",
        "source": "Variety - First Reactions",
        "url": (
            "https://variety.com/2026/film/global/"
            "the-devil-wears-prada-2-first-reactions-"
            "hathaway-streep-1236442908/"
        ),
        "date": "2026-04-29",
        "stage": "first_reactions"
    },

    "Toy Story 5": {
        "score": 2,
        "label": "strong_positive",
        "confidence": "high",
        "source": "Variety - First Reactions",
        "url": (
            "https://variety.com/2026/film/global/"
            "toy-story-5-first-reactions-1236460158/"
        ),
        "date": "2026-06-10",
        "stage": "first_reactions"
    },

    "GOAT": {
        "score": 0,
        "label": "mixed",
        "confidence": "low",
        "source": "Sony Pictures Animation",
        "url": (
            "https://www.linkedin.com/company/"
            "sony-pictures-animation/"
        ),
        "date": "2025-12-01",
        "stage": "trailer_buzz"
    },

    "Backrooms": {
        "score": 1,
        "label": "positive",
        "confidence": "high",
        "source": "Variety - Pre-Release Review",
        "url": (
            "https://variety.com/2026/film/news/"
            "backrooms-review-chiwetel-ejiofor-"
            "kane-parsons-1236453861/"
        ),
        "date": "2026-05-28",
        "stage": "pre_release_review"
    },

    # FUTURE PREDICTION MOVIES

    "The Cat in the Hat": {
        "score": 0,
        "label": "neutral",
        "confidence": "low",
        "source": "Variety - Trailer Coverage",
        "url": (
            "https://variety.com/2025/film/news/"
            "cat-in-the-hat-trailer-bill-hader-"
            "dr-seuss-1236443039/"
        ),
        "date": "2025-07-02",
        "stage": "trailer"
    },

    "Dune: Part Three": {
        "score": 2,
        "label": "strong_positive",
        "confidence": "high",
        "source": "Variety - CinemaCon / Trailer Coverage",
        "url": (
            "https://variety.com/2026/film/news/"
            "dune-3-cinemacon-opening-scene-"
            "footage-chalamet-1236432580/"
        ),
        "date": "2026-04-15",
        "stage": "cinemacon_footage"
    },

    "Avengers: Doomsday": {
        "score": 2,
        "label": "strong_positive",
        "confidence": "high",
        "source": "Variety - CinemaCon Trailer Coverage",
        "url": (
            "https://variety.com/2026/film/news/"
            "avengers-doomsday-trailer-robert-downey-"
            "jr-doctor-doom-1236433230/"
        ),
        "date": "2026-04-17",
        "stage": "cinemacon_footage"
    },

    "Shrek 5": {
        "score": -1,
        "label": "negative",
        "confidence": "medium",
        "source": "ComingSoon - Fan Reaction Analysis",
        "url": (
            "https://www.comingsoon.net/movies/features/"
            "2150432-why-shrek-5-trailers-new-"
            "animation-style-is-dividing-fans"
        ),
        "date": "2026-06-18",
        "stage": "trailer_reaction"
    },

    "Jumanji: Open World": {
        "score": 1,
        "label": "positive",
        "confidence": "medium",
        "source": "Variety - CinemaCon Coverage",
        "url": (
            "https://variety.com/2026/film/news/"
            "jumanji-3-title-open-world-trailer-"
            "cinemacon-robin-williams-tribute-1236432112/"
        ),
        "date": "2026-04-14",
        "stage": "cinemacon_footage"
    },

    "Street Fighter": {
        "score": 1,
        "label": "positive",
        "confidence": "medium",
        "source": "GamesRadar - Trailer Reaction",
        "url": (
            "https://www.gamesradar.com/entertainment/"
            "live-action-movies/the-new-street-fighter-"
            "trailer-is-now-playable-and-good-luck-"
            "nailing-those-combos/"
        ),
        "date": "2026-09-02",
        "stage": "trailer_reaction"
    },

    "The Batman Part II": {
        "score": np.nan,
        "label": "insufficient_evidence",
        "confidence": "low",
        "source": "Variety",
        "url": (
            "https://variety.com/2026/film/news/"
            "the-batman-part-2-scarlett-johansson-"
            "sebastian-stan-1236449318/"
        ),
        "date": "2026-05-15",
        "stage": "production_news"
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "score": np.nan,
        "label": "insufficient_evidence",
        "confidence": "low",
        "source": "Variety",
        "url": (
            "https://variety.com/2026/film/news/"
            "spider-man-beyond-spider-verse-"
            "trailer-leak-1236484915/"
        ),
        "date": "2026-08-15",
        "stage": "production_news"
    },

    "Children of Blood and Bone": {
        "score": 0,
        "label": "mixed",
        "confidence": "medium",
        "source": (
            "TheWrap / The Citizen"
        ),
        "url": (
            "https://www.thewrap.com/creative-content/"
            "movies/children-of-blood-and-bone-"
            "cinemacon-footage-description/"
        ),
        "date": "2026-07-29",
        "stage": "trailer_reaction"
    }
}


# APPLY SENTIMENT DATA

for title, info in sentiment_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "pre_release_sentiment"
    ] = info["score"]

    target_data.loc[
        mask,
        "pre_release_sentiment_label"
    ] = info["label"]

    target_data.loc[
        mask,
        "sentiment_confidence"
    ] = info["confidence"]

    target_data.loc[
        mask,
        "sentiment_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "sentiment_source_url"
    ] = info["url"]

    target_data.loc[
        mask,
        "sentiment_as_of"
    ] = info["date"]

    target_data.loc[
        mask,
        "sentiment_stage"
    ] = info["stage"]


# VALIDATION

print("=" * 80)
print("PRE-RELEASE SENTIMENT COLLECTION")
print("=" * 80)

print(
    f"Sentiment scores available: "
    f"{target_data['pre_release_sentiment'].notna().sum()} "
    f"of {len(target_data)}"
)

print(
    f"Insufficient evidence: "
    f"{target_data['pre_release_sentiment'].isna().sum()}"
)


print("\nSENTIMENT LABELS")
print("-" * 40)

print(
    target_data[
        "pre_release_sentiment_label"
    ].value_counts(
        dropna=False
    )
)


display(
    target_data[
        [
            "title",
            "cohort",
            "pre_release_sentiment",
            "pre_release_sentiment_label",
            "sentiment_confidence",
            "sentiment_stage",
            "sentiment_source"
        ]
    ]
)

PRE-RELEASE SENTIMENT COLLECTION
Sentiment scores available: 16 of 19
Insufficient evidence: 3

SENTIMENT LABELS
----------------------------------------
pre_release_sentiment_label
strong_positive          5
positive                 4
mixed                    3
insufficient_evidence    3
negative                 3
neutral                  1
Name: count, dtype: int64


,title,cohort,pre_release_sentiment,pre_release_sentiment_label,sentiment_confidence,sentiment_stage,sentiment_source
0,Spider-Man: Brand New Day,released_backtest,2.0,strong_positive,high,first_reactions,Variety - First Reactions
1,Michael,released_backtest,0.0,mixed,high,pre_release_press,Variety
2,The Odyssey,released_backtest,1.0,positive,medium,premiere_buzz,Variety - World Premiere Coverage
3,Scary Movie,released_backtest,NaN,insufficient_evidence,low,<NA>,<NA>
4,Mortal Kombat II,released_backtest,-1.0,negative,high,pre_release_review,Variety - Pre-Release Review
5,The Super Mario Galaxy Movie,released_backtest,-1.0,negative,high,pre_release_review,Variety - Pre-Release Review
6,The Devil Wears Prada 2,released_backtest,2.0,strong_positive,high,first_reactions,Variety - First Reactions
7,Toy Story 5,released_backtest,2.0,strong_positive,high,first_reactions,Variety - First Reactions
8,GOAT,released_backtest,0.0,mixed,low,trailer_buzz,Sony Pictures Animation
9,Backrooms,released_backtest,1.0,positive,high,pre_release_review,Variety - Pre-Release Review


## 18 PRE-RELEASE AUDIENCE INTEREST PROXY

In [24]:
# ------------------------------------------------------------
# SCALE
#
# 5 = exceptional / event-level interest
# 4 = high interest
# 3 = moderate interest
# 2 = low interest
# 1 = very low interest
#
# IMPORTANT:
# Research-coded proxy based on pre-release visibility,
# franchise awareness, trailer/footage reaction and trade buzz.
# This is NOT a raw social-media engagement count.
# ------------------------------------------------------------

target_data["social_interest_label"] = pd.NA
target_data["social_interest_confidence"] = pd.NA
target_data["social_interest_source"] = pd.NA
target_data["social_interest_notes"] = pd.NA


social_interest_data = {

    "Spider-Man: Brand New Day": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Sony Pictures / Variety",
        "notes": (
            "Major Spider-Man theatrical event with extensive "
            "pre-release coverage and strong first-reaction attention."
        )
    },

    "Michael": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Lionsgate / Variety",
        "notes": (
            "Global Michael Jackson recognition generated very high "
            "awareness ahead of release."
        )
    },

    "The Odyssey": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Universal Pictures / Variety",
        "notes": (
            "Christopher Nolan event film with major ensemble cast "
            "and sustained pre-release attention."
        )
    },

    "Scary Movie": {
        "score": 4,
        "label": "high",
        "confidence": "medium",
        "source": "Paramount Pictures",
        "notes": (
            "Franchise return with legacy cast and nostalgia-driven "
            "pre-release awareness."
        )
    },

    "Mortal Kombat II": {
        "score": 4,
        "label": "high",
        "confidence": "medium",
        "source": "Warner Bros. Pictures",
        "notes": (
            "Recognisable gaming IP and sequel awareness produced "
            "strong genre-audience interest."
        )
    },

    "The Super Mario Galaxy Movie": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Nintendo / Illumination",
        "notes": (
            "Follow-up to a globally recognised Mario film and "
            "video-game property."
        )
    },

    "The Devil Wears Prada 2": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "20th Century Studios / Variety",
        "notes": (
            "Long-awaited sequel with returning principal cast and "
            "substantial nostalgia-driven attention."
        )
    },

    "Toy Story 5": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Disney / Pixar",
        "notes": (
            "Major long-running Pixar franchise with broad "
            "multi-generational recognition."
        )
    },

    "GOAT": {
        "score": 3,
        "label": "moderate",
        "confidence": "medium",
        "source": "Sony Pictures Animation",
        "notes": (
            "Original animated property with notable talent but "
            "lower pre-existing IP recognition."
        )
    },

    "Backrooms": {
        "score": 4,
        "label": "high",
        "confidence": "medium",
        "source": "A24 / Variety",
        "notes": (
            "Large existing online horror following and strong "
            "interest around the feature adaptation."
        )
    },

    "The Cat in the Hat": {
        "score": 3,
        "label": "moderate",
        "confidence": "medium",
        "source": "Warner Bros. Pictures / Variety",
        "notes": (
            "Highly recognisable Dr. Seuss property, though current "
            "film-specific buzz is less intense than major franchises."
        )
    },

    "Dune: Part Three": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Warner Bros. / Legendary / Variety",
        "notes": (
            "Continuation of a major contemporary theatrical franchise "
            "with strongly received CinemaCon material."
        )
    },

    "Avengers: Doomsday": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Marvel Studios / Disney / Variety",
        "notes": (
            "Avengers event film with exceptionally high franchise "
            "awareness and extensive pre-release coverage."
        )
    },

    "Shrek 5": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "DreamWorks Animation / trade coverage",
        "notes": (
            "Long-awaited return of a globally established franchise. "
            "High attention exists despite mixed animation reaction."
        )
    },

    "Jumanji: Open World": {
        "score": 4,
        "label": "high",
        "confidence": "high",
        "source": "Sony Pictures / Variety",
        "notes": (
            "Returning core cast and established franchise generated "
            "strong CinemaCon attention."
        )
    },

    "Street Fighter": {
        "score": 4,
        "label": "high",
        "confidence": "medium",
        "source": "Paramount / Legendary / Capcom",
        "notes": (
            "Globally recognised game franchise with substantial "
            "interest around the new theatrical adaptation."
        )
    },

    "The Batman Part II": {
        "score": 5,
        "label": "exceptional",
        "confidence": "medium",
        "source": "DC Studios / Warner Bros. / Variety",
        "notes": (
            "Highly anticipated continuation of The Batman, although "
            "release remains comparatively distant."
        )
    },

    "Spider-Man: Beyond the Spider-Verse": {
        "score": 5,
        "label": "exceptional",
        "confidence": "high",
        "source": "Sony Pictures Animation / Variety",
        "notes": (
            "Final chapter of a highly visible Spider-Verse storyline "
            "with substantial anticipation around early footage."
        )
    },

    "Children of Blood and Bone": {
        "score": 3,
        "label": "moderate",
        "confidence": "medium",
        "source": "Paramount Pictures / trade coverage",
        "notes": (
            "Established literary audience and growing film awareness, "
            "but less mass-market recognition than major franchises."
        )
    }
}


# APPLY SCORES

for title, info in social_interest_data.items():

    mask = target_data["title"].eq(title)

    target_data.loc[
        mask,
        "social_interest"
    ] = info["score"]

    target_data.loc[
        mask,
        "social_interest_label"
    ] = info["label"]

    target_data.loc[
        mask,
        "social_interest_confidence"
    ] = info["confidence"]

    target_data.loc[
        mask,
        "social_interest_source"
    ] = info["source"]

    target_data.loc[
        mask,
        "social_interest_notes"
    ] = info["notes"]


# VALIDATION

print("=" * 80)
print("PRE-RELEASE AUDIENCE INTEREST")
print("=" * 80)

print(
    f"Interest scores available: "
    f"{target_data['social_interest'].notna().sum()} "
    f"of {len(target_data)}"
)

print("\nINTEREST LEVELS")
print("-" * 40)

print(
    target_data[
        "social_interest_label"
    ].value_counts(
        dropna=False
    )
)


display(
    target_data[
        [
            "title",
            "social_interest",
            "social_interest_label",
            "social_interest_confidence",
            "social_interest_source"
        ]
    ]
)

PRE-RELEASE AUDIENCE INTEREST
Interest scores available: 19 of 19

INTEREST LEVELS
----------------------------------------
social_interest_label
exceptional    11
high            5
moderate        3
Name: count, dtype: int64


,title,social_interest,social_interest_label,social_interest_confidence,social_interest_source
0,Spider-Man: Brand New Day,5.0,exceptional,high,Sony Pictures / Variety
1,Michael,5.0,exceptional,high,Lionsgate / Variety
2,The Odyssey,5.0,exceptional,high,Universal Pictures / Variety
3,Scary Movie,4.0,high,medium,Paramount Pictures
4,Mortal Kombat II,4.0,high,medium,Warner Bros. Pictures
5,The Super Mario Galaxy Movie,5.0,exceptional,high,Nintendo / Illumination
6,The Devil Wears Prada 2,5.0,exceptional,high,20th Century Studios / Variety
7,Toy Story 5,5.0,exceptional,high,Disney / Pixar
8,GOAT,3.0,moderate,medium,Sony Pictures Animation
9,Backrooms,4.0,high,medium,A24 / Variety


## 19 FINAL DATA COLLECTION VALIDATION

In [ ]:
print("=" * 80)
print("FINAL PREDICTION TARGET DATA VALIDATION")
print("=" * 80)


# CORE COLLECTION FIELDS

core_fields = [
    "title",
    "cohort",
    "release_date",
    "release_year",
    "studio_distributor",
    "director",
    "genre",
    "production_company",
    "lead_cast",
    "existing_ip",
    "is_franchise",
    "is_sequel"
]


print("\nCORE DATA COMPLETENESS")
print("-" * 50)

for column in core_fields:

    available = (
        target_data[column]
        .notna()
        .sum()
    )

    print(
        f"{column:30}: "
        f"{available:2} / "
        f"{len(target_data)}"
    )


# NUMERIC / OPTIONAL FEATURES

optional_fields = [
    "budget",
    "franchise_gap_years",
    "director_track_record",
    "cast_track_record",
    "pre_release_sentiment",
    "social_interest",
    "worldwide_box_office"
]


print("\nOPTIONAL / MODEL-CANDIDATE FEATURES")
print("-" * 50)

for column in optional_fields:

    available = (
        target_data[column]
        .notna()
        .sum()
    )

    missing = (
        target_data[column]
        .isna()
        .sum()
    )

    print(
        f"{column:30}: "
        f"{available:2} available | "
        f"{missing:2} missing"
    )


# DUPLICATE CHECK

duplicate_titles = (
    target_data[
        "title"
    ].duplicated().sum()
)


# COHORT CHECK

released_count = (
    target_data[
        "cohort"
    ].eq(
        "released_backtest"
    ).sum()
)

future_count = (
    target_data[
        "cohort"
    ].eq(
        "future_prediction"
    ).sum()
)


print("\nDATASET STRUCTURE")
print("-" * 50)

print(
    f"Total movies:             "
    f"{len(target_data)}"
)

print(
    f"Unique movies:            "
    f"{target_data['title'].nunique()}"
)

print(
    f"Duplicate titles:         "
    f"{duplicate_titles}"
)

print(
    f"Released/backtest:        "
    f"{released_count}"
)

print(
    f"Future prediction:        "
    f"{future_count}"
)


# OUTCOME LEAKAGE CHECK

future_outcomes = (
    target_data.loc[
        target_data[
            "cohort"
        ].eq(
            "future_prediction"
        ),
        "worldwide_box_office"
    ]
    .notna()
    .sum()
)


print("\nLEAKAGE CHECK")
print("-" * 50)

print(
    f"Future movies with box-office outcome: "
    f"{future_outcomes}"
)


if (
    duplicate_titles == 0
    and future_outcomes == 0
):

    print(
        "\nFINAL VALIDATION: PASS"
    )

else:

    print(
        "\nFINAL VALIDATION: REVIEW REQUIRED"
    )

FINAL PREDICTION TARGET DATA VALIDATION

CORE DATA COMPLETENESS
--------------------------------------------------
title                         : 19 / 19
cohort                        : 19 / 19
release_date                  : 19 / 19
release_year                  : 19 / 19
studio_distributor            : 19 / 19
director                      : 19 / 19
genre                         : 19 / 19
production_company            : 19 / 19
lead_cast                     : 19 / 19
existing_ip                   : 19 / 19
is_franchise                  : 19 / 19
is_sequel                     : 19 / 19

OPTIONAL / MODEL-CANDIDATE FEATURES
--------------------------------------------------
budget                        : 10 available |  9 missing
franchise_gap_years           : 14 available |  5 missing
director_track_record         : 10 available |  9 missing
cast_track_record             : 19 available |  0 missing
pre_release_sentiment         : 16 available |  3 missing
social_interest            

## 20 SAVING THE COMPLETE DATASET AND SOURCE REGISTER

In [ ]:
# OUTPUT PATHS

COMPLETED_TARGET_PATH = (
    TARGET_DIR
    / "prediction_targets_complete.csv"
)

SOURCE_REGISTER_PATH = (
    TARGET_DIR
    / "prediction_target_source_register.csv"
)


# SAVE COMPLETE TARGET DATA

target_data.to_csv(
    COMPLETED_TARGET_PATH,
    index=False
)

# BUILD SOURCE REGISTER

source_columns = [
    column
    for column in target_data.columns
    if (
        "source" in column.lower()
        or "url" in column.lower()
    )
]


source_records = []


for _, row in target_data.iterrows():

    title = row["title"]

    for column in source_columns:

        value = row[column]

        if pd.notna(value):

            source_records.append({
                "title": title,
                "source_field": column,
                "source": value
            })


source_register = pd.DataFrame(
    source_records
).drop_duplicates()


# ADD PRIMARY WEBSITES USED DURING COLLECTION

primary_websites = pd.DataFrame({

    "title": ["PROJECT SOURCE"] * 18,

    "source_field": [
        "website"
    ] * 18,

    "source": [
        "Sony Pictures - https://www.sonypictures.com/",
        "Sony Pictures Animation - https://www.sonypicturesanimation.com/",
        "Disney - https://www.disney.com/",
        "Marvel - https://www.marvel.com/",
        "Pixar - https://www.pixar.com/",
        "20th Century Studios - https://www.20thcenturystudios.com/",
        "Universal Pictures - https://www.universalpictures.com/",
        "DreamWorks - https://www.dreamworks.com/",
        "Warner Bros. - https://www.warnerbros.com/",
        "DC - https://www.dc.com/",
        "Paramount - https://www.paramount.com/",
        "Legendary - https://www.legendary.com/",
        "Nintendo - https://www.nintendo.com/",
        "A24 - https://a24films.com/",
        "Lionsgate - https://www.lionsgate.com/",
        "Variety - https://variety.com/",
        "The Numbers - https://www.the-numbers.com/",
        "Rotten Tomatoes - https://www.rottentomatoes.com/"
    ]
})


source_register = pd.concat(
    [
        source_register,
        primary_websites
    ],
    ignore_index=True
)


source_register.to_csv(
    SOURCE_REGISTER_PATH,
    index=False
)


# CONFIRM OUTPUTS

print("=" * 80)
print("NOTEBOOK 03 OUTPUTS SAVED")
print("=" * 80)

print(
    "\nCompleted prediction dataset:"
)

print(
    COMPLETED_TARGET_PATH
)


print(
    "\nSource register:"
)

print(
    SOURCE_REGISTER_PATH
)


print(
    f"\nDataset rows: "
    f"{len(target_data)}"
)

print(
    f"Source records: "
    f"{len(source_register):,}"
)

NOTEBOOK 03 OUTPUTS SAVED

Completed prediction dataset:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets\prediction_targets_complete.csv

Source register:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets\prediction_target_source_register.csv

Dataset rows: 19
Source records: 286


## 21 DATA COLLECTION REPORT AND NOTEBOOK CLOSURE

In [ ]:
REPORT_PATH = (
    TARGET_DIR
    / "03_prediction_target_data_collection_report.md"
)


# SUMMARY STATISTICS

total_movies = len(
    target_data
)

released_movies = (
    target_data[
        "cohort"
    ]
    .eq(
        "released_backtest"
    )
    .sum()
)

future_movies = (
    target_data[
        "cohort"
    ]
    .eq(
        "future_prediction"
    )
    .sum()
)

budget_available = (
    target_data[
        "budget"
    ]
    .notna()
    .sum()
)

sentiment_available = (
    target_data[
        "pre_release_sentiment"
    ]
    .notna()
    .sum()
)

interest_available = (
    target_data[
        "social_interest"
    ]
    .notna()
    .sum()
)

box_office_available = (
    target_data[
        "worldwide_box_office"
    ]
    .notna()
    .sum()
)


# BUILD REPORT

report = f"""
# Prediction Target Data Collection Report

## Project

**Predicting Movie Box Office Success Using Machine Learning**

## Stage

Prediction Target Data Collection

## Purpose

This stage created a structured dataset for the movies selected for
box-office prediction and backtesting.

The collection process focused primarily on information that would
reasonably be available before theatrical release.

## Prediction Pool

A total of **{total_movies} movies** were retained.

- Released / backtest movies: **{released_movies}**
- Future prediction movies: **{future_movies}**

The released cohort will later be used to compare model predictions
against observed worldwide theatrical performance.

The future cohort represents the films for which final box-office
predictions will be generated.

## Data Collected

The collection stage gathered or derived:

- Release dates and release years
- Studios and distributors
- Production companies
- Genres
- Directors
- Lead cast
- Production budgets where credibly available
- Franchise and sequel status
- Existing-IP classification
- Franchise gap years
- Release timing
- Director historical track record
- Cast historical track record
- Pre-release sentiment
- Pre-release audience-interest proxy
- Worldwide box-office outcomes for released backtest films

## Data Availability

Production budget available:

**{budget_available} / {total_movies} movies**

Pre-release sentiment available:

**{sentiment_available} / {total_movies} movies**

Audience-interest proxy available:

**{interest_available} / {total_movies} movies**

Worldwide box-office outcome available:

**{box_office_available} / {total_movies} movies**

## Source Strategy

Official studio, distributor and production-company sources were
preferred wherever possible.

Major sources included:

- Sony Pictures
- Sony Pictures Animation
- Disney
- Marvel Studios
- Pixar
- 20th Century Studios
- Universal Pictures
- DreamWorks Animation
- Warner Bros.
- DC Studios
- Paramount Pictures
- Legendary Entertainment
- Nintendo
- A24
- Lionsgate

Reputable industry and box-office sources were used where official
sources did not provide sufficient information, including:

- Variety
- The Numbers
- Rotten Tomatoes

A machine-readable source register was saved alongside the completed
prediction-target dataset.

## Data Quality Decisions

Unknown values were retained as missing rather than estimated without
credible evidence.

Reported and estimated budgets were labelled separately.

Future movies contain no worldwide box-office outcome values.

Pre-release sentiment and audience-interest scores were collected
without using post-release audience or box-office results.

## Modelling Caution

Pre-release sentiment and audience-interest variables are currently
candidate features.

They should only be included in the final machine-learning model if
comparable historical versions can be created for the model-training
dataset. Otherwise they may be retained for descriptive analysis and
prediction interpretation.

## Excluded / Later-Bin Targets

The following previously considered films were removed from the active
prediction pool:

- Narnia
- Untitled Paranormal Activity film
- Star Wars: The Mandalorian and Grogu

These were excluded because the available information or theatrical
comparability was considered insufficient for the current modelling
objective.

## Outputs

The stage produced:

1. `prediction_targets_complete.csv`
2. `prediction_target_source_register.csv`
3. `03_prediction_target_data_collection_report.md`

## Stage Status

**Prediction Target Data Collection: COMPLETE**

## NextUp

Proceed to **Notebook 04 — Feature Engineering**.

The next stage will transform the collected historical and target data
into consistent numerical and categorical features suitable for
machine-learning models.
"""


# SAVE REPORT

REPORT_PATH.write_text(
    report.strip(),
    encoding="utf-8"
)


# DISPLAY CLOSURE

print("=" * 80)
print("NOTEBOOK 03 - PREDICTION TARGET DATA COLLECTION")
print("=" * 80)

print(
    "\nSTATUS: COMPLETE"
)

print(
    f"\nMovies collected: "
    f"{total_movies}"
)

print(
    f"Released/backtest: "
    f"{released_movies}"
)

print(
    f"Future predictions: "
    f"{future_movies}"
)

print(
    "\nReport saved to:"
)

print(
    REPORT_PATH
)

print(
    "\nNEXTUP:"
)

print(
    "Notebook 04 - Feature Engineering"
)

NOTEBOOK 03 - PREDICTION TARGET DATA COLLECTION

STATUS: COMPLETE

Movies collected: 19
Released/backtest: 10
Future predictions: 9

Report saved to:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets\03_prediction_target_data_collection_report.md

NEXTUP:
Notebook 04 - Feature Engineering
